# Lab: Mitigate Overfitting
## Purpose:
- Understand how applying weight decay, dropout, and early stopping affects a model's ability to generalize.

### Topics:
- Weight Decay
- Dropout
- Early Stopping

### Steps
* Load the dataset of 2-dimensional embeddings.
* Set different hyperparameters.
* Train the models and plot the learning curve and decision boundary.

Date: 2026-02-21

Source: https://colab.research.google.com/github/google-deepmind/ai-foundations/blob/master/course_3/gdm_lab_3_6_mitigate_overfitting.ipynb

References: https://github.com/google-deepmind/ai-foundations
- GDM GH repo used in AI training courses at the university & college level.

In [ ]:
%%capture
# Install the custom package for this course.
!pip install "git+https://github.com/google-deepmind/ai-foundations.git@main"

import os # For adjusting Keras settings.
os.environ["KERAS_BACKEND"] = "jax" # Set a parameter for Keras.

# Packages used.
from collections import namedtuple # For storing hyperparameter configs.
import keras # For defining your MLP.
import jax.numpy as jnp # For defining matrices.
import pandas as pd # For loading the dataset.
# For splitting data into train and test splits.
from sklearn import model_selection

from ai_foundations import machine_learning # For training your MLP.
from ai_foundations import visualizations # For visualizing data and boundaries.
from ai_foundations import training # For logging the loss during training.

### Load the Dataset

In [ ]:
# Load data using pandas.
df = pd.read_csv("https://storage.googleapis.com/dm-educational/assets/ai_foundations/mat-apple-bank-dataset.csv")

# Extract embeddings (Embedding_dim_1, Embedding_dim_2) and labels.
X = jnp.array(df[["Embedding_dim_1", "Embedding_dim_2"]].values)
# Labels: 0 ("mat"), 1 ("apple"), or 2 ("bank").
y = jnp.array(df["Label"].values)

# Human-readable labels.
labels = ["mat", "apple", "bank"]

# Manually split the data.
X_train, X_test, y_train, y_test = X[0:61, :], X[61:, :], y[0:61], y[61:]

## An MLP with dropout

Dropout is applied *only during training*, and therefore when you generate predictions from the model, you have to specify whether this prediction happens during training or during testing.

Dropout works by randomly setting neurons to 0, which makes the network more robust to differences in input.

The model below adds dropout to the output of each hidden layer using `keras.layers.Dropout()`.
The dropout rate (`dropout_rate`) defines the probability with which a neuron's value is set to 0.

In [ ]:
def build_mlp(
    hidden_dims: list[int], n_classes: int, dropout_rate: float
) -> keras.Model:
    """
    Builds an MLP with a SoftMax layer as the output layer.
    Dropout is applied after each hidden layer.

    Args:
      hidden_dims: A list of dimensions for all hidden layers.
      n_classses: Number of classes for the output layer.
      dropout_rate: Dropout rate, the probability with which a neuron's value is
        set to 0.

    Returns:
      A list of operations in the form of Keras Layer instances.
    """
    operations = []
    # OOF: dim, ReLU, dropout
    for dim in hidden_dims:
        operations.append(keras.layers.Dense(dim))
        operations.append(keras.layers.Activation("relu"))
        operations.append(keras.layers.Dropout(dropout_rate))

    # Apply the bias after the layers have all completed.
    operations.append(keras.layers.Dense(n_classes))  # Output layer: n_classes.
    operations.append(keras.layers.Softmax())

    return keras.Sequential(operations)

## Tune hyperparameters

Change one hyperparameters one at a time.

Train and evaluate a model for each set of hyperparameters, then inspect the loss curves and accuracy curves for each training run.

In [ ]:
# This is just configuring a named tuple for storing hyperparameter configurations.
HyperParameterConfig = namedtuple(
    "HyperParameterConfig",
    [
        "hidden_dims",
        "dropout_rate",
        "weight_decay_strength",
        "use_early_stopping",
    ],
)

# Dictionary stores the results.
experiment_log = {}

def train_and_visualize(
    hidden_dims: tuple[int, ...] | list[int],
    dropout_rate: float,
    weight_decay_strength: float,
    use_early_stopping: bool,
):
    keras.utils.set_random_seed(124)  # For reproducibility.
    model = build_mlp(list(hidden_dims), 3, dropout_rate)

    # This is new. What is it?
    loss_fn = keras.losses.SparseCategoricalCrossentropy()

    learning_rate = 0.01
    optimizer = keras.optimizers.Adam(
        learning_rate=learning_rate, weight_decay=weight_decay_strength
    )

    model.compile(optimizer=optimizer, loss=loss_fn, metrics=["accuracy"])

    epochs = 400
    # Set the patience parameter for early stopping.
    # The model will stop training when the test loss goes up for this many epochs.
    patience = 20

    training_logger = training.CustomAccuracyPrinter(print_every=20)

    callbacks = [training_logger]
    if use_early_stopping:
        callbacks.append(
            keras.callbacks.EarlyStopping(
                monitor="val_loss", verbose=1, patience=patience
            )
        )

    # Train the model with early stopping.
    training_history = model.fit(
        X_train,
        y_train,
        validation_data=(X_test, y_test),
        epochs=epochs,
        verbose=0,
        callbacks=callbacks,
    )

    # Plot loss curves and accuracy curves.
    visualizations.plot_loss_curve(training_history.history)
    visualizations.plot_accuracy_curve(training_history.history)

    visualizations.plot_data_and_mlp(
        X_train,
        y_train,
        labels=labels,
        features_test=X_test,
        label_ids_test=y_test,
        mlp_model=model,
        title="Decision Boundaries",
    )

    print(f"Final test loss: {training_history.history['val_loss'][-1]}")
    print(
        f"Final test accuracy:"
        f" {training_history.history['val_accuracy'][-1]*100:.2f}%"
    )

### The baseline

Every other run will be compared to this set of parameters.

*It is set to overfit so I can experiment with weight decay, dropout, and early stopping.*
*The baseline will have an accuracy of 83.33%.*

In [ ]:
hyperparameters_baseline = HyperParameterConfig(
    hidden_dims=(10, 1000), # Define MLP with many parameters.
    dropout_rate=0.0, # Dropout rate of 0.0 means no dropout is applied.
    weight_decay_strength=0.00, # Value of 0.0 means no weight decay.
    use_early_stopping=False # Early stopping is not enabled.
)

train_and_visualize(*hyperparameters_baseline)

### Documenting experiment results.

------
> **Experiment log**
>```
> # Add the final results of each run to the log so I can compare them later and pick the best result set.
>experiment_log = {}
------

# Save the final test loss from the baseline experiment.
experiment_log[hyperparameters_baseline] = 0.7096465229988098

### Early stopping

------
>Enter the hyperparameters below, but change `use_early_stopping = True`.
------
*The accuracy will be 100%.*

In [ ]:
# Set the hyperparameters here.
hyperparameters_early_stopping = HyperParameterConfig(
    hidden_dims=(10, 1000), # Define MLP with many parameters.
    dropout_rate=0.0, # Dropout rate of 0.0 means no dropout is applied.
    weight_decay_strength=0.00, # Value of 0.0 means no weight decay.
    use_early_stopping= True
)

train_and_visualize(*hyperparameters_early_stopping)

### Dropout

------
>The loss and accuracy will spike as different neurons are dropped.
>Higher dropout = high-amplitude spikes.
------
*Medium dropout (0.3) accuracy will be 91.67%.*

*High dropout (0.7) accuracy will be 58.33%.*

In [ ]:
# Medium Dropout
hyperparameters_early_stopping = HyperParameterConfig(
    hidden_dims=(10, 1000), # Define MLP with many parameters.
    dropout_rate=0.3, # Dropout rate of 0.0 means no dropout is applied.
    weight_decay_strength=0.00, # Value of 0.0 means no weight decay.
    use_early_stopping= False
)

train_and_visualize(*hyperparameters_early_stopping)

In [ ]:
# High Dropout
hyperparameters_early_stopping = HyperParameterConfig(
    hidden_dims=(10, 1000), # Define MLP with many parameters.
    dropout_rate=0.7, # Dropout rate of 0.0 means no dropout is applied.
    weight_decay_strength=0.00, # Value of 0.0 means no weight decay.
    use_early_stopping= False
)

train_and_visualize(*hyperparameters_early_stopping)

In [ ]:
experiment_log[hyperparameters_early_stopping] =0.08434680849313736
experiment_log[hyperparameters_dropout_medium] =0.07518089562654495
experiment_log[hyperparameters_dropout_high] =2.4573543071746826

### Decay strength

------
>Set the weight decay strength. The training procedure will penalize weights that are large magnitude and assign a lower loss to simpler models.
------
*Medium decay strength (0.5) accuracy will be 91.67%%.*

*High decay strength (50) accuracy will be 50.00%%%.*


In [ ]:
# Medium decay
hyperparameters_weight_decay_medium = HyperParameterConfig(
    hidden_dims=(10, 1000), # Define MLP with many parameters.
    dropout_rate=0.0, # Dropout rate of 0.0 means no dropout is applied.
    weight_decay_strength=0.50, # Value of 0.0 means no weight decay.
    use_early_stopping= False
)

train_and_visualize(*hyperparameters_weight_decay_medium)

In [ ]:
# High decay
hyperparameters_weight_decay_medium = HyperParameterConfig(
    hidden_dims=(10, 1000), # Define MLP with many parameters.
    dropout_rate=0.0, # Dropout rate of 0.0 means no dropout is applied.
    weight_decay_strength=50, # Value of 0.0 means no weight decay.
    use_early_stopping= False
)

train_and_visualize(*hyperparameters_weight_decay_medium)

In [ ]:
experiment_log[hyperparameters_weight_decay_medium] =0.42485305666923523
experiment_log[hyperparameters_weight_decay_high] =1.0826328992843628

# Plot the experiment log
visualizations.visualize_hyperparameter_loss(experiment_log)